# Avance 5 - Modelo final: mejora del mejor modelo (Equipo 17, AgroSatCopilot)

## Proyecto Integrador MNA - Tec de Monterrey

**Equipo 17**

- Carlos Isaac Avila Gutierrez - A01796035
- Carlos Aaron Bocanegra Buitron - A01796345
- Arthur Jafed Zizumbo Velasco - A01796363

**Curso**: MNA - Tec de Monterrey - 20-abr -> 3-jul-2026

**Sponsor academico**: Dr. Gerardo Jose Camacho - gjcamacho@tec.mx

**Fecha de entrega**: 2026-06-07.

---

## Resumen ejecutivo

El Avance 4 comparo **6 arquitecturas** de segmentacion densa sobre PASTIS-R y selecciono **TSViT-pheno** como mejor modelo individual (mIoU 0.625, F1-macro 0.750, pixel-acc 0.876, fold-4). Este cuaderno **mejora ese modelo** atacando la causa raiz de su brecha a produccion: el **desbalance de clases** (la diferencia pixel-acc 0.876 vs F1-macro 0.750 indica que las clases minoritarias hunden el promedio macro).

Plan completo, lifts ajustados y compuerta de produccion: [plan Avance 5](../../docs/us-planning/avance5-mejora-modelo-final.md).

## Objetivos y rubrica del Avance 5

- **Ensambles (60 pts)**: 4 ensambles homogeneos y heterogeneos (EPIC 6) - en cuaderno hermano `ensembles/`.
- **Seleccion del modelo final (20 pts)**: tabla comparativa baseline vs mejorado.
- **Graficas interpretadas (>=4, 20 pts)**: curvas, IoU por clase, matriz de confusion, comparativa.

> **Veredicto honesto** (verificado contra el codigo y el calendario): el target F1>=0.80 / mIoU>=0.70 en **flat-18 NO se alcanza** solo con el modelo individual; el cierre del ultimo tramo es trabajo de los ensambles (EPIC 6). La decision de produccion (Avance 6) se toma por **GO-condicional + doble taxonomia** (flat-18 + grouped-6 HCAT) con fallback al baseline XGB+AlphaEarth, no declarando victoria sobre el umbral.

## Como correr en Colab (GPU)

El entrenamiento corre en el **servidor de Colab**, no en tu maquina:

1. `Entorno de ejecucion -> Cambiar tipo de entorno -> GPU` (L4 / A100 en Colab Pro; T4 en free).
2. Ejecuta las celdas en orden: la siguiente **monta Drive, clona el repo e instala dependencias** automaticamente (repo privado -> pide token una vez).
3. Los **datos PASTIS-R y los artefactos viven en el Drive compartido** (`MyDrive/Integrador/`); no se copian al repo.
4. Pon `RUN_TRAINING=True` para lanzar el entrenamiento. En **L4 con batch 16** ronda **~1 h** (40 ep; el run de TSViT del Avance 4 fueron 30 ep en ~32 min en una RTX 4070); puede variar segun el I/O de Drive. El checkpoint se guarda en Drive y la corrida es reanudable.

In [ ]:
# --- Bootstrap Colab: monta Drive, clona el repo, instala deps ---
import os, subprocess, sys
from pathlib import Path

_IN_COLAB = False
shared_folder_path = ''
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shared_folder_path = '/content/drive/MyDrive/Integrador/'
    _IN_COLAB = True
except ImportError:
    pass

# En Colab el repo no esta presente: se clona una vez.
if _IN_COLAB:
    from getpass import getpass
    _repo_dir = '/content/agrosat-copilot'
    _branch = 'user/abocanegra/semana-5'  # branch con la mejora del modelo
    _repo = 'github.com/ArthurZizumbo/agrosat-copilot.git'
    if not Path(_repo_dir, 'pyproject.toml').is_file():
        _rc = os.system(
            f'git clone --branch {_branch} --depth 1 '
            f'https://{_repo} {_repo_dir}')
        if _rc != 0:  # repo privado: pide token (no se guarda)
            _tok = getpass('GitHub token (repo privado): ')
            os.system(
                f'git clone --branch {_branch} --depth 1 '
                f'https://{_tok}@{_repo} {_repo_dir}')

# Localiza el repo por su pyproject.toml y entra en el.
_search = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if _IN_COLAB:
    _search = [Path('/content/agrosat-copilot'), *_search]
for _cand in _search:
    if (_cand / 'pyproject.toml').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        os.chdir(_cand)
        break
else:
    raise RuntimeError('No se encontro el repo agrosat-copilot.')

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'segmentation-models-pytorch', 'structlog', 'typer',
                    'polars', 'mlflow', 'optuna'], check=False)

print('repo:', Path.cwd(), '| colab:', _IN_COLAB,
      '| drive:', shared_folder_path or '(local)')

In [ ]:
# --- Configuracion de la corrida (datos y artefactos en Drive) ---
import matplotlib.pyplot as plt  # noqa: F401
import polars as pl

_base = shared_folder_path if shared_folder_path else ''
# PASTIS-R sigue en Drive (no esta en DVC: no hay data/PASTIS-R.dvc).
PASTIS_ROOT = Path(_base + 'data/PASTIS-R')
# Avance 4 (SOLO lectura): los parquets HOMOLOGADOS estan en GIT (repo),
# no en Drive. Arthur los forzo a git con import_avance4_from_drive.py
# 'para que sea reproducible desde un clon limpio' (commit c223b06). Las
# copias de Drive son las viejas por-integrante (esquemas distintos) que
# rompen el concat: por eso se lee del repo.
A4_METRICS = Path('reports/segmentation/metrics')
# Entregable Avance 5 (escritura): cada etapa con sus propios archivos.
STAGE_DIR = Path(_base + 'reports/best_model')
METRICS_DIR = STAGE_DIR / 'metrics'
FIGURES_DIR = STAGE_DIR / 'figures'
CHECKPOINT_DIR = STAGE_DIR / 'checkpoints'
for _d in (METRICS_DIR, FIGURES_DIR, CHECKPOINT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
MLFLOW_URI = 'file:' + str(STAGE_DIR / 'mlruns')

RUN_NAME = 'alt-tsvit-pheno-cw-aug-v1'  # variante mejorada
BASE_RUN_NAME = 'alt-tsvit-pheno-v1'    # baseline del Avance 4
EPOCHS = 40                             # best del Avance 4 fue 28/30
DEVICE = 'cuda' if _IN_COLAB else 'auto'
RUN_TRAINING = False                    # True para entrenar (horas)

print('PASTIS_ROOT:', PASTIS_ROOT, '| existe:', PASTIS_ROOT.exists())
print('artefactos Avance 5:', STAGE_DIR)
print('device     :', DEVICE, '| run:', RUN_NAME)

## 1. Punto de partida: el mejor modelo del Avance 4

TSViT-pheno gano por amplio margen (los temporales doblan a los spatial-only). La tabla del Avance 4 confirma la seleccion y cuantifica la brecha a los targets de produccion (F1>=0.80, mIoU>=0.70).

In [ ]:
# --- Recap Avance 4: tabla comparativa + brecha del mejor modelo ---
parts = sorted(A4_METRICS.glob('model_comparison_avance4_*.parquet'))
# Cada integrante exporto su parquet con un esquema distinto (distinto
# numero de columnas), por eso se proyecta a un set comun y se concatena
# con diagonal_relaxed (une por nombre y rellena faltantes con null).
_keep = ['model', 'miou', 'f1_macro', 'pixel_accuracy',
         'miou_grouped', 'f1_macro_grouped', 'pixel_accuracy_grouped']
_frames = []
for _p in parts:
    _d = pl.read_parquet(_p)
    _frames.append(_d.select([c for c in _keep if c in _d.columns]))
if _frames:
    a4 = (pl.concat(_frames, how='diagonal_relaxed')
            .unique(subset=['model'], keep='last')
            .sort('miou', descending=True, nulls_last=True))
    display(a4)
    best = a4.row(0, named=True)
    print(f"Mejor: {best['model']} | mIoU {best['miou']:.3f} "
          f"| F1-macro {best['f1_macro']:.3f}")
    print(f"Brecha -> F1: {0.80 - best['f1_macro']:+.3f} "
          f"| mIoU: {0.70 - best['miou']:+.3f}")
else:
    print('No hay parquets de Avance 4 en', A4_METRICS)

## 2. Estrategia de mejora (sin cambiar la arquitectura)

La brecha es desbalance puro, asi que las palancas atacan el F1-macro de las minoritarias, no la arquitectura (que ya es la mejor):

| Palanca | Que hace | Flag CLI |
|---------|----------|----------|
| Class-weighted Dice+CE | pondera CE por clase (effective-number) | `--class-balance effective` |
| Augmentation D4 | flips/rot90 sincronizados (regulariza) | `--augment` |
| 40 ep + early-stopping | el best fue 28/30 (no convergio) | `--epochs 40 --patience 8` |

Codigo: `ml/train/train_segmentation.py` (`_resolve_class_weights`) y `ml/data/pastis_seg_dataset.py` (`apply_synchronized_augment`).

> Los pesos por clase se computan **solo sobre los folds de train** (sin leakage). Su cache es efimero (vive en el repo de la sesion, se recalcula si reinicias); los **resultados reutilizables** (checkpoint, MLflow, parquets, figuras) se guardan en `reports/best_model/` en el Drive compartido.

In [ ]:
# --- Lanzar el entrenamiento de la variante mejorada (subprocess) ---
cmd = [
    sys.executable, '-m', 'ml.train.train_segmentation',
    '--model', 'tsvit-pheno',
    '--target', 'semantic18',
    '--epochs', str(EPOCHS),
    '--patience', '8',
    '--batch-size', '16',
    '--n-timesteps', '10',
    '--device', DEVICE,
    '--root', str(PASTIS_ROOT),
    '--ckpt-dir', str(CHECKPOINT_DIR / RUN_NAME),
    '--mlflow-uri', MLFLOW_URI,
    '--augment',
    '--class-balance', 'effective',
    '--class-balance-beta', '0.9999',
    '--run-name', RUN_NAME,
]
print('comando:', ' '.join(cmd))
if RUN_TRAINING:
    proc = subprocess.run(cmd)
    print('returncode:', proc.returncode)
else:
    print('RUN_TRAINING=False -> no entrena. Pon True en Colab con GPU.')

## 3. Comparativa baseline vs mejorado (seleccion - 20 pts)

Lee las metricas de validacion (fold-4) de ambos runs desde MLflow y las compara. El modelo final es el mejor de los dos por mIoU; el delta cuantifica el aporte de las palancas anti-desbalance.

In [ ]:
# --- Comparativa baseline (Avance 4) vs mejorado (stage Avance 5) ---
rows = []
# Baseline tsvit-pheno: del parquet del Avance 4 (entregable anterior).
_a4 = A4_METRICS / 'model_comparison_avance4_tsvit.parquet'
if _a4.exists():
    _b = pl.read_parquet(_a4).filter(pl.col('model') == 'tsvit-pheno')
    if _b.height:
        _r = _b.row(0, named=True)
        rows.append({'run': BASE_RUN_NAME, 'miou': _r['miou'],
                     'f1_macro': _r['f1_macro'],
                     'pixel_acc': _r['pixel_accuracy']})
# Run mejorado: del MLflow de este stage (reports/best_model/mlruns).
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_URI)
    df = mlflow.search_runs(search_all_experiments=True)
    name_col = 'tags.mlflow.runName'
    if name_col in df.columns:
        for _, r in df[df[name_col] == RUN_NAME].iterrows():
            rows.append({
                'run': RUN_NAME,
                'miou': r.get('metrics.best_miou', r.get('metrics.miou')),
                'f1_macro': r.get('metrics.best_f1_macro',
                                  r.get('metrics.f1_macro')),
                'pixel_acc': r.get('metrics.best_pixel_acc',
                                   r.get('metrics.pixel_acc')),
            })
except Exception as exc:  # noqa: BLE001
    print('MLflow no disponible:', exc)

if rows:
    comp = pl.DataFrame(rows).sort('miou', descending=True, nulls_last=True)
    display(comp)
    _out = METRICS_DIR / 'avance5_best_model_comparison.parquet'
    comp.write_parquet(str(_out))
    print('escrito', _out)
else:
    print('Pendiente: corre el entrenamiento (RUN_TRAINING=True).')

## 4. Error analysis per-clase (calibracion + model card)

El F1-macro se hunde por unas pocas clases minoritarias. Esta tabla (recall / precision / F1 / soporte por cultivo) las identifica y alimenta el model card del Avance 6. Usa `DenseConfusionAccumulator.per_class_metrics` sobre el fold-4 con el checkpoint del modelo final.

In [ ]:
# --- Tabla per-clase del modelo final (degrada si falta ckpt/datos) ---
import torch
from torch.utils.data import DataLoader

from ml.data.pastis_seg_dataset import PASTISSegmentationDataset
from ml.eval.dense_metrics import DenseConfusionAccumulator

CKPT = CHECKPOINT_DIR / RUN_NAME / 'best.pt'
try:
    if not CKPT.exists():
        raise FileNotFoundError(CKPT)
    from ml.models.tsvit_wrapper import build_tsvit
    model = build_tsvit(num_classes=18, n_timesteps=10, img_size=128,
                        in_channels=10, semantic_dim=384)
    state = torch.load(CKPT, map_location='cpu')
    model.load_state_dict(state.get('model', state))
    dev = 'cuda' if torch.cuda.is_available() and DEVICE != 'cpu' else 'cpu'
    model = model.to(dev).eval()
    val_ds = PASTISSegmentationDataset(
        root=PASTIS_ROOT, folds=(4,), collapse_time=None,
        n_timesteps=10, target='semantic18')
    acc = DenseConfusionAccumulator(num_classes=18, ignore_index=255)
    with torch.no_grad():
        for x, y in DataLoader(val_ds, batch_size=4):
            out = model(x.to(dev))
            logits = out[0] if isinstance(out, tuple) else out
            acc.update(logits.argmax(1).cpu(), y)
    per_class = pl.DataFrame(acc.per_class_metrics()).sort('f1')
    display(per_class)
    per_class.write_parquet(
        str(METRICS_DIR / 'avance5_per_class_tsvit_pheno.parquet'))
except Exception as exc:  # noqa: BLE001
    print('Pendiente (corre el entrenamiento primero):', exc)

## 5. Graficas interpretadas (>=4, 20 pts)

Curvas de entrenamiento, IoU por clase, matriz de confusion y comparativa baseline vs mejorado. Se reutilizan los helpers de `ml/eval/avance4_figures.py` y las figuras exportadas a `reports/segmentation/figures/`.

In [ ]:
# --- Galeria de figuras del modelo final (degrada con placeholder) ---
from IPython.display import Image, Markdown, display

_fig_types = [('curves', 'Curvas de entrenamiento'),
              ('per_class_iou', 'IoU por clase'),
              ('confusion', 'Matriz de confusion'),
              ('samples', 'RGB / verdad / prediccion')]
_models = ('tsvit-pheno-cw-aug', 'tsvit-pheno', 'tsvit_pheno')
_shown = False
for _key, _label in _fig_types:
    for _model in _models:
        _f = FIGURES_DIR / f'{_key}_{_model}.png'
        if _f.exists():
            display(Markdown(f'**{_label}** ({_model})'))
            display(Image(filename=str(_f)))
            _shown = True
            break
if not _shown:
    display(Markdown('_Pendiente: genera las figuras tras el entrenamiento._'))

## 6. Modelo final y compuerta de produccion (Avance 6)

**Modelo final**: TSViT-pheno + class-weights + augmentation (variante `alt-tsvit-pheno-cw-aug-v1`), seleccionado por mIoU/F1-macro de validacion.

**Lectura honesta de produccion** (no se fuerza el numero):

- En **flat-18** el modelo individual no cruza 0.80/0.70; lo acerca y el resto lo cierran los ensambles (EPIC 6).
- En **grouped-6 HCAT** (taxonomia agronomica que el stakeholder consume) la metrica es mas alta; se reporta junto al flat-18, nunca como sustituto.
- La decision GO / GO-condicional / NO-GO se toma con el **model card** + **arbol de contingencia** con fallback al baseline XGB+AlphaEarth (F1>=0.60 garantizado), y despliegue via Pub/Sub + Cloud Run L4 worker (regla global 9).

Detalle: [plan Avance 5, seccion 5](../../docs/us-planning/avance5-mejora-modelo-final.md).

**Proximos pasos**: 4 ensambles obligatorios (cuaderno hermano), 3-fold CV del config ganador para el numero defendible con intervalos de confianza.